In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import random
import pickle
import requests
import zipfile
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.sparse.linalg import eigsh
from scipy.sparse import csgraph
from scipy.linalg import eigh
import networkx as nx

In [ ]:
# based on experimentation with the graphs
networks_of_interest = [1, 11, 19, "g21", 7]

In [ ]:
# --- Configuration ---
URL = "https://snap.stanford.edu/data/CiscoSecureWorkload_22_networks.zip"

# Detect if we are in /notebooks/ or the root project folder
cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

# Standardize the data paths relative to the project root
data_dir = project_root / "data"
CACHE_FILE = data_dir / "cisco_adj_matrices.pkl"
expected_root = data_dir / "Cisco_22_networks"
zip_path = data_dir / "CiscoSecureWorkload_22_networks.zip"

# --- Logic: Determine if we can skip ---
root = None

if CACHE_FILE.exists():
    print(f"Cache file found at {CACHE_FILE}. Skipping download.")
    root = expected_root
elif expected_root.exists():
    print(f"Raw data found at {expected_root}. Skipping download.")
    root = expected_root
else:
    print(f"No data found. Preparing to download into {data_dir}...")
    data_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        r = requests.get(URL, stream=True)
        r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print("Extracting files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
            
        os.remove(zip_path)
        root = expected_root
        print("Done!")
    except Exception as e:
        print(f"Error: {e}")

# Set root for the rest of your notebook
root = str(root) if root else None
print(f"Root path set to: {root}")

In [ ]:
def get_all_matrices(root_path, output_file, force_rebuild=False):
    """
    Builds or loads all 22+ Cisco matrices. 
    Reports summary stats sorted by Flatness.
    """
    if os.path.exists(output_file) and not force_rebuild:
        print(f"Loading matrices from {output_file}...")
        with open(output_file, 'rb') as f:
            master_matrices = pickle.load(f)
    else:
        print("Building matrices from raw files (this may take a few minutes)...")
        tasks = []
        for i in range(1, 21):
            tasks.append({'id': i, 'label': f'g{i}', 'filter': f'g{i}', 'dir': "dir_20_graphs"})
        tasks.append({'id': 'g21', 'label': 'g21', 'filter': 'g21', 'dir': "dir_g21_small_workload_with_gt/dir_includes_packets_and_other_nodes"})
        tasks.append({'id': 'g21 subset', 'label': 'g21 subset', 'filter': 'g21', 'dir': "dir_g21_small_workload_with_gt/dir_no_packets_etc"})
        tasks.append({'id': 'g22', 'label': 'g22', 'filter': 'g22', 'dir': "dir_g22_extra_graph_with_gt/dir_edges"})

        master_matrices = {}
        for task in tasks:
            target_path = os.path.join(root_path, task['dir'])
            if not os.path.exists(target_path): continue
            
            files = glob.glob(os.path.join(target_path, "**", "*.gz"), recursive=True)
            all_edges_list = []
            for f in files:
                try:
                    df = pd.read_csv(f, sep=r'\s+', header=None, comment='#', usecols=[0, 1, 2], dtype={0: str, 1: int, 2: int}, engine='python')
                    mask = df[0].str.strip() == task['filter']
                    edges = df.loc[mask, [1, 2]].values
                    if edges.size > 0:
                        u, v = edges[:, 0], edges[:, 1]
                        valid = u != v
                        norm = np.column_stack([np.minimum(u[valid], v[valid]), np.maximum(u[valid], v[valid])])
                        all_edges_list.append(norm)
                except: continue
            
            if all_edges_list:
                unique_edges = np.unique(np.vstack(all_edges_list), axis=0)
                nodes = np.unique(unique_edges)
                n = len(nodes)
                node_map = {node_id: i for i, node_id in enumerate(nodes)}
                u_idx = np.array([node_map[x] for x in unique_edges[:, 0]])
                v_idx = np.array([node_map[x] for x in unique_edges[:, 1]])
                adj = csr_matrix((np.ones(len(u_idx)*2), (np.concatenate([u_idx, v_idx]), np.concatenate([v_idx, u_idx]))), shape=(n, n))
                adj.data[:] = 1 
                master_matrices[task['id']] = adj

        with open(output_file, 'wb') as f:
            pickle.dump(master_matrices, f)

    # Generate Summary Report
    summary = []
    for gid, adj in master_matrices.items():
        n = adj.shape[0]
        e = int(adj.sum() / 2)
        f_g = e / (n * (n - 1) / 2) if n > 1 else 0
        summary.append({"ID": gid, "Nodes": n, "Edges": e, "Flatness": f_g, "Segmentedness": 1-f_g})
    
    df_report = pd.DataFrame(summary).sort_values(by="Flatness", ascending=False)
    print("\nGlobal Network Summary (Sorted by Flatness):")
    display(df_report)
    
    return master_matrices

In [ ]:
def run_monte_carlo(adj_matrix, num_trials=1000, samples_per_trial=97):
    seed = 1
    random.seed(seed)
    np.random.seed(seed)
    
    num_nodes = adj_matrix.shape[0]
    trial_results = []
    for _ in range(num_trials):
        successes = sum(1 for _ in range(samples_per_trial) if adj_matrix[tuple(random.sample(range(num_nodes), 2))] == 1)
        trial_results.append(successes / samples_per_trial)
    return np.mean(trial_results), 1.96 * (np.std(trial_results, ddof=1) / np.sqrt(num_trials))

In [ ]:
def generate_latex(results_list):
    latex = [r"\begin{table}[htbp]", r"\centering", 
             r"\caption{Network Metrics and Estimator Performance for the five densest graphs from Cisco Secure Workload dataset ($M=97$, $1,000$ trials)}",
             r"\label{tab:cisco-results}",
             r"\footnotesize", r"\begin{tabular*}{\columnwidth}{@{\extracolsep{\fill}}lrrrrr@{}}", r"\toprule",
             r"\textbf{Graph} & \textbf{$n$} & \textbf{$|E|$} & \textbf{$F(G)$} & \textbf{Mean $\hat{F}(G)$} & \textbf{95\% CI} \\", r"\midrule"]
    for r in results_list:
        latex.append(f"{r['name']:<10} & {r['n']:>5d} & {r['e']:>7d} & {r['f']:>8.3f} & {r['m']:>10.3f} & [{r['m']-r['moe']:.3f}, {r['m']+r['moe']:.3f}] \\\\")
    latex.extend([r"\bottomrule", r"\end{tabular*}", r"\end{table}"])
    return "\n".join(latex)

In [ ]:
# 1. Load/Build Matrices and show summary
# can take several minutes if a CACHE_FILE is not already present
all_matrices = get_all_matrices(root, CACHE_FILE)

In [ ]:
# 2. Run Simulation for Networks of Interest
sim_data = []
for nid in networks_of_interest:
    if nid in all_matrices:
        adj = all_matrices[nid]
        n, e = adj.shape[0], int(adj.sum() / 2)
        f_true = e / (n * (n - 1) / 2)
        mean_f, moe = run_monte_carlo(adj)
        sim_data.append({
            'name': f"g{nid}" if not str(nid).startswith('g') else nid, 
            'n': n, 
            'e': e, 
            'f': f_true, 
            'm': mean_f, 
            'moe': moe
        })

In [ ]:
# 3. Output LaTeX
print("\nGenerated LaTeX Table:")
print(generate_latex(sim_data))

In [ ]:
def calculate_modularity(adj_matrix):
    """
    Calculates modularity using greedy modularity maximization.
    """
    if adj_matrix.shape[0] < 2:
        return 0.0
    
    # Convert CSR matrix to NetworkX Graph
    G = nx.from_scipy_sparse_array(adj_matrix)
    
    # Use the community sub-module via the main networkx namespace
    from networkx.algorithms import community
    communities = community.greedy_modularity_communities(G)
    
    # Calculate modularity of this partition
    mod_value = community.modularity(G, communities)
    return float(mod_value)
    
def calculate_fiedler_value(adj_matrix, use_lcc=False):
    """
    Calculates the Fiedler value (2nd smallest eigenvalue of the Laplacian).
    
    Parameters:
    adj_matrix: scipy.sparse matrix
    use_lcc (bool): If True, calculates the Fiedler value only for the 
                    Largest Connected Component.
    """
    # OPTIONAL: Extract Largest Connected Component
    if use_lcc:
        G = nx.from_scipy_sparse_array(adj_matrix)
        if not nx.is_connected(G):
            # Get the largest connected component
            largest_cc = max(nx.connected_components(G), key=len)
            G = G.subgraph(largest_cc).copy()
            adj_matrix = nx.to_scipy_sparse_array(G)
            
    n = adj_matrix.shape[0]
    if n < 2:
        return 0.0
    
    try:
        L = csgraph.laplacian(adj_matrix, normed=False)
        
        if n <= 3:
            L_dense = L.toarray() if hasattr(L, 'toarray') else L
            eigvals = eigh(L_dense, eigvals_only=True)
            fiedler = float(np.sort(eigvals)[1])
        else:
            eigvals = eigsh(L.astype(float), k=2, sigma=-1e-5, which='LM', return_eigenvectors=False)
            fiedler = float(np.sort(eigvals)[1])
            
        # Clean up floating point noise (Snap to 0.0 if extremely small)
        if abs(fiedler) < 1e-10:
            return 0.0
            
        return fiedler
            
    except Exception:
        return 0.0

In [ ]:
def display_comprehensive_metrics(all_matrices, subset=None):
    """
    Processes either a specific subset or all graphs.
    """
    results = []
    # If no subset is provided, use all. Otherwise, use the provided list.
    target_ids = subset if subset is not None else all_matrices.keys()
    
    print(f"Calculating metrics for {len(target_ids)} networks...")
    
    for nid in target_ids:
        if nid not in all_matrices:
            continue
            
        adj = all_matrices[nid]
        n = adj.shape[0]
        e = int(adj.sum() / 2)
        
        # Flatness F(G)
        possible = (n * (n - 1)) / 2 if n > 1 else 0
        f_g = e / possible if possible > 0 else 0
        
        # Structural Metrics (Heavy lifting)
        mod_val = calculate_modularity(adj)
        fiedler_val = calculate_fiedler_value(adj, use_lcc=True)
        
        display_id = f"g{nid}" if not str(nid).startswith('g') else nid
        
        results.append({
            "Graph ID": display_id,
            "Flatness F(G)": f_g,
            "Modularity": mod_val,
            "Fiedler Value": fiedler_val
        })
            
    return pd.DataFrame(results)

In [ ]:
def generate_latex_correlation_table(metrics_df):
    """
    LaTeX table without nodes/edges, formatted for advanced metrics.
    """
    latex = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Graph-measures comparison: Flatness, Modularity, and Fiedler Value of the ten densest networks in the Cisco Secure Workload dataset}",
        r"\label{tab:structural-correlation}",
        r"\footnotesize",
        r"\begin{tabular*}{\columnwidth}{@{\extracolsep{\fill}}lrrr@{}}",
        r"\toprule",
        r"\textbf{Graph} & \textbf{$F(G)$} & \textbf{Modularity} & \textbf{Fiedler Value} \\",
        r"\midrule"
    ]

    df_sorted = metrics_df.sort_values(by="Flatness F(G)", ascending=False)

    for _, row in df_sorted.iterrows():
        fiedler_str = f"{row['Fiedler Value']:.2e}" if row['Fiedler Value'] < 0.001 else f"{row['Fiedler Value']:.3f}"
        line = (
            f"{row['Graph ID']:<12} & "
            f"{row['Flatness F(G)']:>8.3f} & "
            f"{row['Modularity']:>10.3f} & "
            f"{fiedler_str:>10} \\\\"
        )
        latex.append(line)

    latex.extend([r"\bottomrule", r"\end{tabular*}", r"\end{table}"])
    return "\n".join(latex)

In [ ]:
# --- EXECUTION ---
# this cell takes a few seconds to run

# 1. Re-define your target networks
ten_densest = [1, 11, 19, "g21", 7, 18, 16, 17, 14, 8]

# 2. Generate metrics ONLY for the networks of interest (faster than all networks)
interest_metrics_df = display_comprehensive_metrics(all_matrices, subset=ten_densest)

# 3. Calculate Absolute Correlation Coefficients 
corr_matrix = interest_metrics_df[['Flatness F(G)', 'Modularity', 'Fiedler Value']].corr().abs()
flat_mod_corr = corr_matrix.loc['Flatness F(G)', 'Modularity']
flat_fiedler_corr = corr_matrix.loc['Flatness F(G)', 'Fiedler Value']

# 4. Display Results
print("\n--- Correlation Analysis ---")
print(f"Absolute Correlation |Flatness, Modularity|: {flat_mod_corr:.3f}")
print(f"Absolute Correlation |Flatness, Fiedler|:    {flat_fiedler_corr:.3f}")

print("\n--- Summary Table ---")
display(interest_metrics_df.sort_values("Flatness F(G)", ascending=False))

# 5. Generate LaTeX
print("\n--- Generated LaTeX (No Nodes/Edges) ---")
print(generate_latex_correlation_table(interest_metrics_df))

In [ ]:
# Sanity Checking

print(f"{'Graph ID':<10} | {'Components':<12} | {'Status'}")
print("-" * 40)

for nid in ten_densest:
    # Safe lookup: check the first key, if not found, check the alternative
    adj = None
    if nid in all_matrices:
        adj = all_matrices[nid]
    else:
        # Try stripping 'g' if it's a string, or adding it
        alt_id = str(nid).replace('g', '')
        if alt_id in all_matrices:
            adj = all_matrices[alt_id]
        elif f"g{nid}" in all_matrices:
            adj = all_matrices[f"g{nid}"]
    
    if adj is not None:
        # Calculate components
        n_components, labels = connected_components(csgraph=adj, directed=False, return_labels=True)
        
        status = "Connected" if n_components == 1 else "DISCONNECTED"
        display_id = f"g{nid}" if not str(nid).startswith('g') else nid
        
        print(f"{display_id:<10} | {n_components:<12} | {status}")
    else:
        print(f"{nid:<10} | {'N/A':<12} | Not found in all_matrices")